In [1]:
!unzip "/content/EESA-Corpus-main.zip" -d "/content/EESA"
!ls /content/EESA

Archive:  /content/EESA-Corpus-main.zip
c98f360aa121a26ead8b23db61e78e60d72cca60
   creating: /content/EESA/EESA-Corpus-main/
  inflating: /content/EESA/EESA-Corpus-main/EESA-Dev.csv  
  inflating: /content/EESA/EESA-Corpus-main/EESA-Lexicon.csv  
  inflating: /content/EESA/EESA-Corpus-main/EESA-Test.csv  
  inflating: /content/EESA/EESA-Corpus-main/EESA-Train.csv  
  inflating: /content/EESA/EESA-Corpus-main/README.md  
EESA-Corpus-main


In [4]:
!ls /content

EESA  EESA-Corpus-main.zip  sample_data


In [5]:
!ls /content/EESA

EESA-Corpus-main


In [6]:
!ls /content/EESA/EESA-Corpus-main

EESA-Dev.csv  EESA-Lexicon.csv	EESA-Test.csv  EESA-Train.csv  README.md


In [10]:
# PyTorch + HuggingFace Framework

import time
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from torch.nn import CrossEntropyLoss


# Reproducibility
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


# Load CSV datasets
train_df = pd.read_csv("/content/EESA/EESA-Corpus-main/EESA-Train.csv")
val_df = pd.read_csv("/content/EESA/EESA-Corpus-main/EESA-Dev.csv")
test_df = pd.read_csv("/content/EESA/EESA-Corpus-main/EESA-Test.csv")

# renaming columns for consistency
train_df.columns = ["text", "label"]
val_df.columns = ["text", "label"]
test_df.columns = ["text", "label"]

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))


# label format (convert to integers)

print("Unique labels before mapping:", train_df["label"].unique())

label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

train_df["label"] = train_df["label"].map(label_map)
val_df["label"] = val_df["label"].map(label_map)
test_df["label"] = test_df["label"].map(label_map)

train_df["label"] = train_df["label"].astype(int)
val_df["label"] = val_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

print("Unique labels after mapping:", train_df["label"].unique())


# Convert to HuggingFace dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)


# Tokenization
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)


# set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


# Class weights (for imbalance)
labels = np.array(train_df["label"])
num_labels = len(np.unique(labels))

class_counts = np.bincount(labels)
class_weights = len(labels) / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class weights:", class_weights)


# Custom Trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fn = CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits.view(-1, num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


# Evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_weighted": precision_w,
        "recall_weighted": recall_w,
        "f1_weighted": f1_w,
        "precision": precision_m,
        "recall": recall_m,
        "f1": f1_m
    }


# Helper functions
def build_model():
    return AutoModelForSequenceClassification.from_pretrained(
        "xlm-roberta-base",
        num_labels=num_labels
    )


def evaluate_model(trainer, dataset, name):
    results = trainer.evaluate(eval_dataset=dataset)

    preds_output = trainer.predict(dataset)
    y_true = preds_output.label_ids
    y_pred = np.argmax(preds_output.predictions, axis=-1)

    print(f"\n{name} Classification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    print(f"{name} Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    return results, y_true, y_pred


# Training arguments
full_args = TrainingArguments(
    output_dir="./full_model",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    report_to="none"
)

lora_args = TrainingArguments(
    output_dir="./lora_model",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    report_to="none"
)


# Full Fine-Tuning
print("\n----- Full Fine-Tuning -----")

full_model = build_model()

full_trainer = WeightedTrainer(
    model=full_model,
    args=full_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

start_time = time.time()
full_trainer.train()
full_time = time.time() - start_time

full_val_metrics, _, _ = evaluate_model(full_trainer, val_dataset, "Validation")
full_test_metrics, _, _ = evaluate_model(full_trainer, test_dataset, "Test")


# LoRA Fine-Tuning
print("\n----- LoRA Fine-Tuning -----")

base_model = build_model()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    task_type=TaskType.SEQ_CLS,
    target_modules=["query", "value"],
    modules_to_save=["classifier"]
)

lora_model = get_peft_model(base_model, lora_config)

lora_trainer = WeightedTrainer(
    model=lora_model,
    args=lora_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

start_time = time.time()
lora_trainer.train()
lora_time = time.time() - start_time

lora_val_metrics, _, _ = evaluate_model(lora_trainer, val_dataset, "Validation")
lora_test_metrics, _, _ = evaluate_model(lora_trainer, test_dataset, "Test")


# Comparison
comparison = pd.DataFrame([
    {
        "Approach": "Full Fine-Tuning",
        "Training Time": round(full_time, 2),
        "Val F1": round(full_val_metrics["eval_f1"], 4),
        "Test F1": round(full_test_metrics["eval_f1"], 4)
    },
    {
        "Approach": "LoRA",
        "Training Time": round(lora_time, 2),
        "Val F1": round(lora_val_metrics["eval_f1"], 4),
        "Test F1": round(lora_test_metrics["eval_f1"], 4)
    }
])

print("\n----- Comparison -----")
print(comparison)

Device: cuda
Train size: 2463
Validation size: 817
Test size: 817
Unique labels before mapping: ['neutral' 'negative' 'positive']
Unique labels after mapping: [1 0 2]


Map:   0%|          | 0/2463 [00:00<?, ? examples/s]

Map:   0%|          | 0/817 [00:00<?, ? examples/s]

Map:   0%|          | 0/817 [00:00<?, ? examples/s]

Class weights: tensor([1.3845, 1.0553, 0.7518], device='cuda:0')

----- Full Fine-Tuning -----


...

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted,Precision,Recall,F1
1,No log,0.720031,0.778458,0.777759,0.778458,0.774854,0.767974,0.765463,0.763325


Epoch,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted,Precision,Recall,F1
1,No log,0.720031,0.778458,0.777759,0.778458,0.774854,0.767974,0.765463,0.763325
2,0.804256,0.529719,0.804162,0.811611,0.804162,0.802691,0.791867,0.800290,0.789708
3,0.804256,0.507875,0.832313,0.836336,0.832313,0.833651,0.820056,0.827823,0.823270
4,0.539828,0.501898,0.826193,0.829256,0.826193,0.826590,0.812784,0.821468,0.815692
5,0.484300,0.522544,0.834761,0.835726,0.834761,0.834874,0.823576,0.829355,0.825996



Validation Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.84      0.81       196
           1       0.80      0.77      0.78       258
           2       0.89      0.88      0.89       363

    accuracy                           0.83       817
   macro avg       0.82      0.83      0.83       817
weighted avg       0.84      0.83      0.83       817

Validation Confusion Matrix:
[[165  21  10]
 [ 32 198  28]
 [ 14  30 319]]



Test Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.77      0.78       197
           1       0.76      0.83      0.79       258
           2       0.93      0.88      0.90       362

    accuracy                           0.84       817
   macro avg       0.83      0.83      0.83       817
weighted avg       0.84      0.84      0.84       817

Test Confusion Matrix:
[[151  35  11]
 [ 29 215  14]
 [ 10  34 318]]

----- Comparison -----
           Approach  Training Time  Val F1  Test F1
0  Full Fine-Tuning         618.49  0.8349   0.8341
1              LoRA         239.01  0.8260   0.8253
